In [ ]:
'''if 'google.colab' in str(get_ipython()):
    ! git clone -b main https://github.com/edsonportosilva/OptiCommPy
    from os import chdir as cd
    cd('/content/OptiCommPy/')
    ! pip install .'''

import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))


In [ ]:
from tqdm.notebook import tqdm
import numpy as np
import cupy as cp;
import numpy.lib.stride_tricks
import cupy.lib.stride_tricks
from numpy.fft import fft, ifft, fftfreq
from  scipy.constants import c
import matplotlib.pyplot as plt
import time
from Algoritmo.equalization import mimoAdaptEq, resultados

from optic.comm.modulation import modulateGray, grayMapping
from optic.dsp.core import pnorm, upsample, firFilter, pulseShape, signalPower, phaseNoise, decimate, symbolSync, finddelay
from optic.dsp.equalization import edc, mimoAdaptEqualizer
from optic.models.devices import iqm, coherentReceiver, pdmCoherentReceiver, basicLaserModel, mzm, edfa
from optic.models.channels import linearFiberChannel, awgn
from optic.utils import parameters
from optic.comm.metrics import fastBERcalc
from optic.plot import pconst, eyediagram, plotPSD
from optic.models.tx import simpleWDMTx
from optic.dsp.carrierRecovery import cpr

In [ ]:
    ############## Canal Óptico ##############
def canal_optico(sigWDM_Tx, paramEDFA, paramTx):

  sigCh       = edfa(sigWDM_Tx, paramEDFA)                                      # modelo linear do EDFA

  # # deteriorates the signal-to-noise ratio before the fiber
  SNR         = 13
  sigAWGN     = awgn(sigWDM_Tx, SNR, Fs, paramTx.Rs)

  # simulate linear signal propagation
  sigCh       = linearFiberChannel(sigWDM_Tx, paramFiber)

  return sigCh

In [ ]:
    ############## Receptor Óptico (PMD)##############
def receptor_optico(sigCh, paramTx, paramPD):

  # parameters
  chIndex     = 0  # index of the channel to be demodulated

  Fc          = paramTx.Fc
  Ts          = 1/Fs
  freqGrid    = paramTx.wdmFreqGrid
  π           = np.pi
  t           = np.arange(0, len(sigCh))*Ts

  symbTx      = symbTx_[:,:,chIndex]

  # local oscillator (LO) parameters:
  FO      = 150e6                                                               # deslocamento de frequência
  Δf_lo   = freqGrid[chIndex]+FO                                                # redução do canal a ser demodulado
  paramLO.Ns = len(sigCh)

  sigLO = basicLaserModel(paramLO)
  sigLO = sigLO*np.exp(1j*2*π*Δf_lo*t)                                          # adiciona deslocamento de frequência

  # polarization multiplexed coherent optical received
  θsig        = π/3 # polarization rotation angle
  sigRx       = pdmCoherentReceiver(sigCh, sigLO, θsig, paramPD)

  return sigRx

In [ ]:
    ############## Filtro Casado e Compensação de CD ##############
def filtro_casado_cd(sigRx, paramTx, paramEDC):

  # Matched filtering
  paramPulse              = parameters()
  paramPulse.pulseType    = paramTx.pulseType
  paramPulse.SpS          = paramTx.SpS
  paramPulse.nFilterTaps  = paramTx.nFilterTaps
  paramPulse.pulseRollOff = paramTx.pulseRollOff

  pulse                   = pulseShape(paramPulse)

  pulse                   = pnorm(pulse)
  pulse                   = cp.asarray(pulse)
  sigRx                   = cp.asarray(sigRx)

  sigRx                   = firFilter(pulse, sigRx)
  sigRx                   = sigRx.get()

  sigRx                   = edc(sigRx, paramEDC)

  # decimate to one sample per symbol
  paramDec                = parameters()
  paramDec.SpSin          = paramTx.SpS
  paramDec.SpSout         = 1
  sigRx                   = decimate(sigRx, paramDec)

  x                       = pnorm(sigRx)
  x                       = cp.asarray(x)

  return x

In [ ]:
    ############## Equalizadores ##############
def equalizador(x, paramTx, Nsymb):

  paramEq               = parameters()
  if (Nsymb == 7e3):
   paramEq.nTaps       = 3
   paramEq.mu          = [5e-1, 1e-2]
   paramEq.N1          = 500
   paramEq.N2          = 3000
   paramEq.blockSize   = 2**6

  elif (Nsymb == 2e4):
   paramEq.nTaps       = 10
   paramEq.mu          = [7e-1, 7e-2]
   paramEq.N1          = 4000
   paramEq.N2          = 15000
   paramEq.blockSize   = 2**9

  elif (Nsymb == 7e4):
   paramEq.nTaps       = 15
   paramEq.mu          = [7e-1, 7e-2]
   paramEq.N1          = 7000
   paramEq.N2          = 20000
   paramEq.blockSize   = 2**10

  elif (Nsymb == 3e5):
   paramEq.nTaps       = 15
   paramEq.mu          = [9e-1, 9e-2]
   paramEq.N1          = 10000
   paramEq.N2          = 25000
   paramEq.blockSize   = 2**11

  elif (Nsymb == 1e6):
   paramEq.nTaps       = 15
   paramEq.mu          = [9e-1, 9e-2]
   paramEq.N1          = 15000
   paramEq.N2          = 30000
   paramEq.blockSize   = 2**11

  else:
   print('Número de símbolos não suportado')
   breakpoint()

  paramEq.M             = paramTx.M
  paramEq.constType     = paramTx.constType
  paramEq.alg           = 'cma-to-rde'
  paramEq.progBar       = True

  t_start1              = time.perf_counter()
  y_EQ                  = mimoAdaptEq(x.get(), paramEq)
  t_end1                = time.perf_counter()
  timeEqCPU             = t_end1 - t_start1

  cp.cuda.Stream.null.synchronize()
  t_start2              = time.perf_counter()
  y_EQ_gpu              = mimoAdaptEq(x, paramEq)
  cp.cuda.Stream.null.synchronize()
  t_end2                = time.perf_counter()
  timeEqGPU             = t_end2 - t_start2

  yEQ                   = y_EQ[0]

  # Parametros da Portadora de Fase
  paramCPR              = parameters()
  paramCPR.alg          = 'bps'
  paramCPR.M            = paramTx.M
  paramCPR.N            = 75
  paramCPR.B            = 64

  y_CPR                 = cpr(yEQ, paramCPR, symbTx_)

  # Equalizador Adaptativo
  paramEq.alg           = 'dd-lms'
  y_CPR_gpu             = cp.asarray(y_CPR)

  t_start3              = time.perf_counter()
  y_LMS, e, _           = mimoAdaptEq(y_CPR, paramEq)
  t_end3                = time.perf_counter()
  timeEqAdCPU           = t_end3 - t_start3

  cp.cuda.Stream.null.synchronize()
  t_start4              = time.perf_counter()
  y_LMS_gpu, e, _       = mimoAdaptEq(y_CPR_gpu, paramEq)
  cp.cuda.Stream.null.synchronize()
  t_end4                = time.perf_counter()
  timeEqAdGPU           = t_end4 - t_start4

  y_CPR_LMS             = cpr(y_LMS, paramCPR)

  return y_CPR, y_CPR_LMS, timeEqCPU, timeEqGPU, timeEqAdCPU, timeEqAdGPU

In [ ]:
t_start = time.perf_counter()
##################### TRANSMISSOR #####################

# Transmitter parameters:
paramTx                 = parameters()
paramTx.M               = 64                                                    # ordem do formato de modulação
paramTx.constType       = 'qam'                                                 # esquema de modulação
paramTx.seed            = 42                                                    # semente - torna saidas no mesmo padrao
paramTx.Rs              = 32e9                                                  # taxa de símbolo [baud]
paramTx.SpS             = 16                                                    # amostras por símbolo
paramTx.pulseType       = 'rrc'                                                 # filtro de modelagem de pulso
paramTx.nFilterTaps     = 4096                                                  # número de coeficientes de filtro de modelagem de pulso
paramTx.pulseRollOff    = 0.01                                                  # Eliminação do RRC
paramTx.powerPerChannel = -2                                                    # potência por canal WDM [dBm]
paramTx.nChannels       = 1                                                     # número de canais WDM
paramTx.Fc              = 193.1e12                                              # frequência óptica central do espectro WDM
paramTx.laserLinewidth  = 100e3                                                 # largura de linha do laser em Hz
paramTx.wdmGridSpacing  = 37.5e9                                                # Espaçamento de grade WDM
paramTx.nPolModes       = 2                                                     # número de modos de sinal [2 para sinais multiplexados por polarização]

Fs                      = paramTx.Rs * paramTx.SpS                              # taxa de amostragem de simulação

## fiber parameters:
paramFiber              = parameters()
paramFiber.L            = 300                                                   # comprimento do enlace [km]
paramFiber.alpha        = 0.2                                                   # coeficiente de perdas [dB/Km]
paramFiber.D            = 17                                                    # parâmetro de dispersão [ps/nm/km]
paramFiber.Fs           = Fs                                                    # Frequência de amostragem do sinal [amostras/segundo]
paramFiber.Fc           = paramTx.Fc                                            # Frequência central do sinal [Hz]

# EDFA parameters:
paramEDFA               = parameters()
paramEDFA.G             = paramFiber.alpha * paramFiber.L                       # Ganho para compensar
paramEDFA.Fs            = Fs
paramEDFA.NF            = 7
paramEDFA.Fc            = paramTx.Fc

# generate CW laser LO field
paramLO                 = parameters()
paramLO.P               = 10                                                    # power in dBm
paramLO.lw              = 100e3                                                 # laser linewidth
paramLO.RIN_var         = 0                                                     # variancia do ruido
paramLO.Fs              = Fs

# Define os parâmetros do receptor coerente
paramPD                 = parameters()
paramPD.B               = paramTx.Rs
paramPD.Fs              = Fs
paramPD.ideal           = True

# CD compensation
paramEDC                = parameters()
paramEDC.L              = paramFiber.L
paramEDC.D              = paramFiber.D
paramEDC.Fc             = paramTx.Fc
paramEDC.Fs             = Fs

numberOfSymbols         = np.array([7e3, 2e4, 7e4, 3e5, 1e6])

timeEqCPU = np.zeros(len(numberOfSymbols))
timeEqGPU = np.zeros(len(numberOfSymbols))
timeEqAdCPU = np.zeros(len(numberOfSymbols))
timeEqAdGPU = np.zeros(len(numberOfSymbols))

for idx, Nsymb in enumerate(tqdm(numberOfSymbols)):

    paramTx.nBits       = int(np.log2(paramTx.M)*Nsymb)                         # total number of bits per polarization

    # generate WDM signal
    sigWDM_Tx, symbTx_, paramTx = simpleWDMTx(paramTx)

    sigCh               = canal_optico(sigWDM_Tx, paramEDFA, paramTx)           # função retorna o campo óptico complexo depois de passar pelo EDFA → AWGN → fibra linear.
    sigRx               = receptor_optico(sigCh, paramTx, paramPD)              # função retorna o sinal elétrico após o fotodetector balanceado após sigCh → LO → mixer → PD → sigRx
    x                   = filtro_casado_cd(sigRx, paramTx, paramEDC)            # função reorna o sinal após o filtro casado, a compensação de dispersão e decimação
    y_CPR, y_CPR_LMS, timeEqCPU_, timeEqGPU_, timeEqAdCPU_, timeEqAdGPU_        = equalizador(x, paramTx, Nsymb)         # função retorna os sinais equalizados apos o CMA → RDE → DD-LMS
    # Armazenando os tempos calculados em cada repetição

    timeEqCPU[idx]      = timeEqCPU_
    timeEqGPU[idx]      = timeEqGPU_
    timeEqAdCPU[idx]    = timeEqAdCPU_
    timeEqAdGPU[idx]    = timeEqAdGPU_
    tx, rx = resultados(y_CPR_LMS, paramTx, symbTx_, Nsymb)

np.savez_compressed(
    "tempo_execucao_colab.npz",
    numberOfSymbols=numberOfSymbols.astype(float),  # garante float
    timeEqCPU=timeEqCPU,
    timeEqGPU=timeEqGPU,
    timeEqAdCPU=timeEqAdCPU,
    timeEqAdGPU=timeEqAdGPU
)
print("Arquivo salvo: tempo_execucao_colab.npz")

In [ ]:
plt.plot(numberOfSymbols, timeEqCPU, '-o', linewidth=1.8, label='CPU')
plt.plot(numberOfSymbols, timeEqGPU, '-s', linewidth=1.8, label='GPU')
plt.xlabel('Length of the signal (number of samples)')
plt.ylabel('Processing time (s)')
plt.legend()
plt.grid()
plt.xlim(min(numberOfSymbols), max(numberOfSymbols));

plt.figure()
plt.plot(numberOfSymbols, timeEqCPU/timeEqGPU,'-o', label='GPU speedup')
plt.xlabel('Length of the signal (number of samples)')
plt.ylabel('GPU speedup ($\\times$)')
plt.legend()
plt.grid()
plt.xlim(min(numberOfSymbols), max(numberOfSymbols));

In [ ]:
plt.plot(numberOfSymbols, timeEqAdCPU, '-o', linewidth=1.8, label='CPU')
plt.plot(numberOfSymbols, timeEqAdGPU, '-s', linewidth=1.8, label='GPU')
plt.xlabel('Length of the signal (number of samples)')
plt.ylabel('Processing time (s)')
plt.legend()
plt.grid()
plt.xlim(min(numberOfSymbols), max(numberOfSymbols));

plt.figure()
plt.plot(numberOfSymbols, timeEqAdCPU/timeEqAdGPU,'-o', label='GPU speedup')
plt.xlabel('Length of the signal (number of samples)')
plt.ylabel('GPU speedup ($\\times$)')
plt.legend()
plt.grid()
plt.xlim(min(numberOfSymbols), max(numberOfSymbols));

In [ ]:
t = np.arange(100) / Fs

fig, axs = plt.subplots(1, 2, figsize=(15, 5), sharey=True);
# Modo X
axs[0].plot(t, rx[:100, 0], label="Rx – Mode 0")
axs[0].plot(t, tx[:100, 0], label="Tx – Mode 0", alpha=0.7)
axs[0].set_title("Polarização X (Mode 0)")
axs[0].set_xlabel("Tempo [s]")
axs[0].set_ylabel("Amplitude")
axs[0].legend()
axs[0].grid(True)

# Modo Y
axs[1].plot(t, rx[:100, 1], label="Rx – Mode 1")
axs[1].plot(t, tx[:100, 1], label="Tx – Mode 1", alpha=0.7)
axs[1].set_title("Polarização Y (Mode 1)")
axs[1].set_xlabel("Tempo [s]")
axs[1].legend()
axs[1].grid(True)

fig.tight_layout()
plt.show()
t_end = time.perf_counter()
elapsed = t_end - t_start
print(f"Tempo total = {elapsed:.2f} s")